In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols
import numpy as np
from scipy import stats
import statsmodels.api as sm
from matplotlib.ticker import MaxNLocator
#from statannot import add_stat_annotation
from datetime import datetime
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import statsmodels.api as sm
from scipy.stats import iqr
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn import model_selection
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.svm import SVR
from sklearn.svm import SVC
from sklearn.linear_model import Lasso

In [ ]:
def isNaN(num):
    return num != num

# Load Data

In [ ]:
# load labels
labels_df = pd.read_csv("${DEMENTIA_DATA_ROOT}\\Brain Age\\dementia_labels\\StudyList_v2.csv")
#labels_df = labels_df[labels_df['Label'] != 'Unlabelled']

In [ ]:
patient_features_table = pd.read_csv("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\patient_features_table.csv")
patient_metadata_table = pd.read_csv("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\patient_metadata_table.csv")

In [ ]:
patient_metadata_table 

In [ ]:
# load microfeatures

infile = open( "${DEMENTIA_DATA_ROOT}\\Brain Age\\Pickles\\EEG_features.p",'rb')
new_dict = pickle.load(infile)
infile.close()
subjects_s = new_dict[0]
mean_features_s = new_dict[1]
EEG_feature_names = list(pd.read_csv("${DEMENTIA_DATA_ROOT}\\Brain Age\\csv_data\\EEG_features_list.csv",header=None)[0])
features_df = pd.DataFrame(data=mean_features_s['W'],columns = ['W_'+ s for s in EEG_feature_names] )
features_df['feature_file'] = subjects_s['W']

df = pd.DataFrame(data=mean_features_s['N1'],columns = ['N1_'+ s for s in EEG_feature_names] )
df['feature_file'] = subjects_s['N1']
features_df = features_df.merge(df,on=['feature_file'])

df = pd.DataFrame(data=mean_features_s['N2'],columns = ['N2_'+ s for s in EEG_feature_names] )
df['feature_file'] = subjects_s['N2']
features_df = features_df.merge(df,on=['feature_file'])

df = pd.DataFrame(data=mean_features_s['N3'],columns = ['N3_'+ s for s in EEG_feature_names] )
df['feature_file'] = subjects_s['N3']
features_df = features_df.merge(df,on=['feature_file'])

df = pd.DataFrame(data=mean_features_s['R'],columns = ['R_'+ s for s in EEG_feature_names] )
df['feature_file'] = subjects_s['R']
features_df = features_df.merge(df,on=['feature_file'])

features_df = features_df[['feature_file'] + list(features_df.columns[:96]) + list(features_df.columns[97:])]
features_df = features_df.drop_duplicates()
features_df = features_df.reset_index(drop=True)
features_df['feature_path']  = features_df['feature_file'] 
features_df['feature_file'] = [ os.path.basename(row['feature_path']) for index,row in features_df.iterrows()]

In [ ]:
data_df = patient_features_table[['FolderName','feature_path','PatientID','MRN','TypeOfTest']].merge(features_df,on=['feature_path'])

In [ ]:
# load macrofeatures
sleep_stats_df = pd.read_csv("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\patient_sleepstats_table.csv")

In [ ]:
# load macrofeatures
sleep_stats_df

In [ ]:
data_df = data_df.merge(sleep_stats_df[['FolderName', 'Age','AHI',
       'TRT', 'TIB', 'TST', 'N1', 'N1p', 'N2', 'N2p', 'N3', 'N3p', 'REM',
       'REMp', 'W', 'SOL', 'REML', 'WASO', 'WASOp', 'SEI']],on=['FolderName'],how='left')

In [ ]:
data_df = data_df.dropna(subset=['TypeOfTest'])

In [ ]:
features_df = features_df.dropna()

In [ ]:
data_df 

# Predict ACE-R

In [ ]:
RPDR_ACER_df = pd.read_excel("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\RPDR_ACER_df_Reviewed.xlsx")
EDW_ACER_df = pd.read_excel("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\EDW_ACER_df_Reviewed.xlsx")
neuropsych_df = pd.read_excel("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\neuropsychiatric_reports_scores.xlsx")

In [ ]:
RPDR_ACER_df = RPDR_ACER_df[RPDR_ACER_df['Reviewed'] == 'Y']

In [ ]:
RPDR_ACER_df = RPDR_ACER_df.merge(patient_metadata_table[['EMPI','PatientID']])
RPDR_ACER_df = RPDR_ACER_df[['PatientID','LMRNote_Date','ACERScore']]


In [ ]:
RPDR_ACER_df.columns = ['PatientID', 'DateOfScore', 'ACE-R']

In [ ]:
RPDR_ACER_df = RPDR_ACER_df.dropna(subset=['ACE-R'])

In [ ]:
RPDR_ACER_df

In [ ]:
EDW_ACER_df = EDW_ACER_df[EDW_ACER_df['Reviewed'] == 'Y']

In [ ]:
EDW_ACER_df = EDW_ACER_df[['PatientID','ContactDTSForNote','ACERScore']]

In [ ]:
EDW_ACER_df.columns = ['PatientID', 'DateOfScore', 'ACE-R']

In [ ]:
EDW_ACER_df = EDW_ACER_df.dropna(subset=['ACE-R'])
EDW_ACER_df

In [ ]:
ACER_df = pd.concat([RPDR_ACER_df,EDW_ACER_df]).drop_duplicates()

In [ ]:
len(ACER_df.PatientID.unique())

In [ ]:
patient_metadata_table

In [ ]:
ACER_df = patient_metadata_table.merge(ACER_df,on=['PatientID'],how='inner')

In [ ]:
ACER_df['dT'] = [ (datetime.strptime(row['DateOfScore'],'%Y-%m-%d')-datetime.strptime(row['DateOfVisit'],'%Y-%m-%d')).days for index,row in ACER_df.iterrows()]
ACER_df['proximity'] = [abs(row['dT']) for index,row in ACER_df.iterrows()]

In [ ]:
ACER_df = ACER_df.sort_values(by=['FolderName','proximity']).drop_duplicates(subset=['FolderName'])

In [ ]:
ACER_df

In [ ]:
data_df

In [ ]:
ACER_df = ACER_df[['FolderName','ACE-R']]

In [ ]:
ACER_data_df = data_df.merge(ACER_df,on=['FolderName'])

In [ ]:
ACER_data_df[ACER_data_df.columns[6:-1]]

In [ ]:
ACER_data_df = ACER_data_df.sort_values(by=['ACE-R'])

In [ ]:
ACER_data_df = ACER_data_df.interpolate(method='nearest')

In [ ]:
ACER_data_df = ACER_data_df.sample(frac=1)

### Univariate feature selection

In [ ]:
X =  ACER_data_df[ACER_data_df.columns[6:-1]][:-test_num ]
y = ACER_data_df[ACER_data_df.columns[-1]][:-test_num ]

In [ ]:
X[X.columns[i]]

In [ ]:
# first do simple univariate feature selection
corrs = []
for i in range(X.shape[1]):
    corrs.append(np.abs(pearsonr(X[X.columns[i]], y.values)[0]))
corrs = np.array(corrs)

features_corr_df = pd.DataFrame(columns=['feature_name','corr'])
features_corr_df['feature_name'] = ACER_data_df.columns[6:-1]
features_corr_df['corr'] = corrs
features_corr_df = features_corr_df.sort_values(by='corr',ascending=False)
features_corr_df[:20]

In [ ]:
len(corrs)

In [ ]:
test_num = 19

In [ ]:
test_num 

In [ ]:
X =  ACER_data_df[ACER_data_df.columns[6:-1]][:-test_num ]
y = ACER_data_df[ACER_data_df.columns[-1]][:-test_num ]
reg = LinearRegression().fit(X, y)
#np.random.normal(size=len(X))

In [ ]:
reg.score(X, y)


In [ ]:
reg.coef_

In [ ]:
reg.intercept_

In [ ]:
y_pred = reg.predict(ACER_data_df[ACER_data_df.columns[6:-1]][-test_num: ])
y_pred

In [ ]:
y_true = ACER_data_df[ACER_data_df.columns[-1]][-test_num: ]
y_true

In [ ]:
r2_score(y_true, y_pred)

In [ ]:
seed = 7
kfold = model_selection.KFold(n_splits=10, random_state=seed)
model = LinearRegression()
scoring = 'neg_mean_absolute_error'
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring=scoring)
print("MAE:", results.mean(), results.std())
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring='r2')
print("R^2:", results.mean(), results.std())

In [ ]:
 results

In [ ]:
print("MAE:", results.mean(), results.std())


In [ ]:
reg.coef_

### Lasso Regression

In [ ]:
test_num =20

In [ ]:
X =  ACER_data_df[ACER_data_df.columns[6:-1]][:-test_num ]
y = ACER_data_df[ACER_data_df.columns[-1]][:-test_num ]
reg = Lasso(alpha=0.1).fit(X, y)
print(reg.score(X, y))
y_pred = reg.predict(ACER_data_df[ACER_data_df.columns[6:-1]][-test_num: ])
print(y_pred)
y_true = ACER_data_df[ACER_data_df.columns[-1]][-test_num: ]
print(list(y_true))
print(r2_score(y_true, y_pred))
print(np.sqrt(mean_squared_error(y_true, y_pred)))

In [ ]:
plt.scatter(y_true, y_pred)
plt.xlabel('True_Score')
plt.ylabel('Pred_Score')
plt.show()

In [ ]:
seed = 15
kfold = model_selection.KFold(n_splits=10, random_state=seed)
model = Lasso(alpha=0.1)
scoring = 'neg_mean_absolute_error'
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring=scoring)
print(results)
print("MAE:", results.mean(), results.std())
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring='r2')
print(results)
print("R^2:", results.mean(), results.std())

# Predict MoCA

In [ ]:
RPDR_MoCA_df = pd.read_excel("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\RPDR_MoCA_list_Reviewed.xlsx")
EDW_MoCA_df = pd.read_excel("${DEMENTIA_DATA_ROOT}\Brain Age\csv_data\MoCA_list_V4_Reviewed.xlsx")


In [ ]:
RPDR_MoCA_df = RPDR_MoCA_df[RPDR_MoCA_df['Correct'] == 'Y']

In [ ]:
RPDR_MoCA_df = RPDR_MoCA_df.merge(patient_metadata_table,on=['EMPI'])

In [ ]:
RPDR_MoCA_df

In [ ]:
RPDR_MoCA_df = RPDR_MoCA_df[['PatientID','LMRNote_Date','MoCAScore']]

In [ ]:
RPDR_MoCA_df.columns = ['PatientID','DateOfScore','MoCA']

In [ ]:
EDW_MoCA_df = EDW_MoCA_df[EDW_MoCA_df['Reviewed'] == 'Y']

In [ ]:
EDW_MoCA_df  = EDW_MoCA_df[['PatientID','ContactDTSForNote','MoCAScore']]

In [ ]:
EDW_MoCA_df.columns = ['PatientID','DateOfScore','MoCA']

In [ ]:
MoCA_df = pd.concat([EDW_MoCA_df,RPDR_MoCA_df])

In [ ]:
MoCA_df = MoCA_df.drop_duplicates()

In [ ]:
MoCA_df

In [ ]:
MoCA_df = patient_metadata_table.merge(MoCA_df,on=['PatientID'],how='inner')

In [ ]:
MoCA_df['dT'] = [ (datetime.strptime(row['DateOfScore'],'%Y-%m-%d')-datetime.strptime(row['DateOfVisit'],'%Y-%m-%d')).days for index,row in MoCA_df.iterrows()]
MoCA_df['proximity'] = [abs(row['dT']) for index,row in MoCA_df.iterrows()]

In [ ]:
MoCA_df = MoCA_df.sort_values(by=['FolderName','proximity']).drop_duplicates(subset=['FolderName'])

In [ ]:
#conditions
#MoCA_df = MoCA_df[MoCA_df['TypeOfTest'].str.contains('Diagnostic|CPAP')]

In [ ]:
MoCA_df = MoCA_df[['FolderName','MoCA']]

In [ ]:
MoCA_data_df = data_df.merge(MoCA_df,on=['FolderName'])

In [ ]:
MoCA_data_df = MoCA_data_df.sort_values(by=['MoCA'])

In [ ]:
MoCA_data_df = MoCA_data_df.interpolate(method='nearest')

In [ ]:
MoCA_data_df = MoCA_data_df.sample(frac=1)

In [ ]:
MoCA_data_df

### Univariate Feature Selection

In [ ]:
X =  MoCA_data_df[MoCA_data_df.columns[6:-1]][:-test_num ]
y = MoCA_data_df[MoCA_data_df.columns[-1]][:-test_num ]

In [ ]:
X[X.columns[i]]

In [ ]:
# first do simple univariate feature selection
corrs = []
for i in range(X.shape[1]):
    corrs.append(np.abs(pearsonr(X[X.columns[i]], y.values)[0]))
corrs = np.array(corrs)

features_corr_df = pd.DataFrame(columns=['feature_name','corr'])
features_corr_df['feature_name'] = ACER_data_df.columns[6:-1]
features_corr_df['corr'] = corrs
features_corr_df = features_corr_df.sort_values(by='corr',ascending=False)
features_corr_df[:20]

### Linear Regression

In [ ]:
test_num = 19

In [ ]:
test_num 

In [ ]:
X =  MoCA_data_df[MoCA_data_df.columns[6:-1]][:-test_num ]
y = MoCA_data_df[MoCA_data_df.columns[-1]][:-test_num ]
reg = LinearRegression().fit(X, y)
#np.random.normal(size=len(X))

In [ ]:
reg.score(X, y)


In [ ]:
reg.coef_

In [ ]:
reg.intercept_

In [ ]:
y_pred = reg.predict(MoCA_data_df[MoCA_data_df.columns[6:-1]][-test_num: ])
y_pred

In [ ]:
y_true = MoCA_data_df[MoCA_data_df.columns[-1]][-test_num: ]
y_true

In [ ]:
r2_score(y_true, y_pred)

In [ ]:
X =  MoCA_data_df[MoCA_data_df.columns[6:-1]][:-test_num ]
y = MoCA_data_df[MoCA_data_df.columns[-1]][:-test_num ]
reg = LinearRegression().fit(X, y)
print(reg.score(X, y))
y_pred = reg.predict(MoCA_data_df[MoCA_data_df.columns[6:-1]][-test_num: ])
print(y_pred)
y_true = MoCA_data_df[MoCA_data_df.columns[-1]][-test_num: ]
print(list(y_true))
print(r2_score(y_true, y_pred))
print(mean_squared_error(y_true, y_pred))

In [ ]:
seed = 20
kfold = model_selection.KFold(n_splits=10, random_state=seed)
model = LinearRegression()
scoring = 'neg_mean_absolute_error'
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring=scoring)
print(results)
print("MAE:", results.mean(), results.std())
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring='r2')
print(results)
print("R^2:", results.mean(), results.std())

In [ ]:
test_num=50

In [ ]:
X =  MoCA_data_df[MoCA_data_df.columns[6:-1]][:-test_num ]
y = MoCA_data_df[MoCA_data_df.columns[-1]][:-test_num ]
reg = SVR(kernel='rbf',gamma='auto').fit(X, y)
print(reg.score(X, y))
y_pred = reg.predict(MoCA_data_df[MoCA_data_df.columns[6:-1]][-test_num: ])
print(y_pred)
y_true = MoCA_data_df[MoCA_data_df.columns[-1]][-test_num: ]
print(list(y_true))
print(r2_score(y_true, y_pred))
print(np.sqrt(mean_squared_error(y_true, y_pred)))

In [ ]:
plt.scatter(y_true, y_pred)
plt.xlabel('True_Score')
plt.ylabel('Pred_Score')
plt.show()

In [ ]:
svr_poly = SVR(kernel='poly',gamma='auto')
svr_rbf = SVR(kernel='rbf',gamma='auto',C=10)
svr_linear = SVR(kernel='linear',gamma='auto')

In [ ]:
seed = 15
kfold = model_selection.KFold(n_splits=10, random_state=seed)
model = svr_rbf
scoring = 'neg_mean_absolute_error'
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring=scoring)
print(results)
print("MAE:", results.mean(), results.std())
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring='neg_mean_squared_error')
print(results)
print("MSE:", results.mean(), results.std())
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring='r2')
print(results)
print("R^2:", results.mean(), results.std())


In [ ]:
seed = 15
kfold = model_selection.KFold(n_splits=10, random_state=seed)
model = svr_linear
scoring = 'neg_mean_absolute_error'
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring=scoring)
print(results)
print("MAE:", results.mean(), results.std())
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring='r2')
print(results)
print("R^2:", results.mean(), results.std())

In [ ]:
seed = 15
kfold = model_selection.KFold(n_splits=10, random_state=seed)
model = svr_poly
scoring = 'neg_mean_absolute_error'
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring=scoring)
print(results)
print("MAE:", results.mean(), results.std())
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring='r2')
print(results)
print("R^2:", results.mean(), results.std())

### Lasso Regression

In [ ]:
test_num =30

In [ ]:
X =  MoCA_data_df[MoCA_data_df.columns[6:-1]][:-test_num ].values
# then standardize the features
Xmean = np.mean(X, axis=0)
Xstd = np.std(X, axis=0)
X = (X-Xmean)/Xstd

y = MoCA_data_df[MoCA_data_df.columns[-1]][:-test_num ]
reg = Lasso(alpha=0.08).fit(X, y)
#reg =ARDRegression().fit(X, y)

print(reg.score(X, y))
y_pred = reg.predict((MoCA_data_df[MoCA_data_df.columns[6:-1]][-test_num:].values-Xmean)/Xstd)
print(y_pred)
y_true = MoCA_data_df[MoCA_data_df.columns[-1]][-test_num: ]
print(list(y_true))
print(r2_score(y_true, y_pred))
print(np.sqrt(mean_squared_error(y_true, y_pred)))

In [ ]:
plt.scatter(y_true, y_pred)
plt.plot([0,30],[0,30])
plt.xlabel('True_Score')
plt.ylabel('Pred_Score')
plt.show()

In [ ]:
seed = 15
kfold = model_selection.KFold(n_splits=10, random_state=seed)
model = Lasso(alpha=0.1)
scoring = 'neg_mean_absolute_error'
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring=scoring)
print(results)
print("MAE:", results.mean(), results.std())
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring='r2')
print(results)
print("R^2:", results.mean(), results.std())


### PCR

# Predict MMSE

In [ ]:
RPDR_MMSE_df = pd.read_excel("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\RPDR_MMSE_list_Reviewed.xlsx")
EDW_MMSE_df = pd.read_excel("${DEMENTIA_DATA_ROOT}\Brain Age\csv_data\MMSE_list_V2_Reviewed.xlsx")


In [ ]:
RPDR_MMSE_df = RPDR_MMSE_df.merge(patient_metadata_table,on=['EMPI'])
RPDR_MMSE_df = RPDR_MMSE_df[['PatientID','LMRNote_Date','MMSEScore']]

In [ ]:
RPDR_MMSE_df.columns = ['PatientID','DateOfScore','MMSE']

In [ ]:
EDW_MMSE_df

In [ ]:
EDW_MMSE_df = EDW_MMSE_df[['PatientID','ContactDTSForNote','MMSEScore']]

In [ ]:
EDW_MMSE_df.columns = ['PatientID','DateOfScore','MMSE']

In [ ]:
MMSE_df = pd.concat([EDW_MMSE_df,RPDR_MMSE_df])

In [ ]:
MMSE_df = MMSE_df.drop_duplicates()

In [ ]:
MMSE_df = patient_metadata_table.merge(MMSE_df,on=['PatientID'])

In [ ]:
MMSE_df['dT'] = [ (datetime.strptime(row['DateOfScore'],'%Y-%m-%d')-datetime.strptime(row['DateOfVisit'],'%Y-%m-%d')).days for index,row in MMSE_df.iterrows()]
MMSE_df['proximity'] = [abs(row['dT']) for index,row in MMSE_df.iterrows()]

In [ ]:
MMSE_df = MMSE_df.sort_values(by=['FolderName','proximity']).drop_duplicates(subset=['FolderName'])

In [ ]:
MMSE_data_df

In [ ]:
#conditions
#MoCA_df = MoCA_df[MoCA_df['TypeOfTest'].str.contains('Diagnostic|CPAP')]

In [ ]:
MMSE_df = MMSE_df[['FolderName','MMSE']]

In [ ]:
MMSE_data_df = data_df.merge(MMSE_df,on=['FolderName'])

In [ ]:
MMSE_data_df = MMSE_data_df.sort_values(by=['MMSE'])

In [ ]:
MMSE_data_df = MMSE_data_df.interpolate(method='nearest')

In [ ]:
MMSE_data_df = MMSE_data_df.sample(frac=1)

### Univariate Feature Selection

In [ ]:
MMSE_data_df

### Univariate feature selection

In [ ]:
X =  MMSE_data_df[MMSE_data_df.columns[6:-1]][:-test_num ]
y = MMSE_data_df[MMSE_data_df.columns[-1]][:-test_num ]

In [ ]:
X[X.columns[i]]

In [ ]:
# first do simple univariate feature selection
corrs = []
for i in range(X.shape[1]):
    corrs.append(np.abs(pearsonr(X[X.columns[i]], y.values)[0]))
corrs = np.array(corrs)

features_corr_df = pd.DataFrame(columns=['feature_name','corr'])
features_corr_df['feature_name'] = MMSE_data_df.columns[6:-1]
features_corr_df['corr'] = corrs
features_corr_df = features_corr_df.sort_values(by='corr',ascending=False)
features_corr_df[:20]

### Lasso Regression

In [ ]:
test_num =30

In [ ]:
len(MMSE_data_df)

In [ ]:
X =  MMSE_data_df[MMSE_data_df.columns[6:-1]][:-test_num ].values
# then standardize the features
Xmean = np.mean(X, axis=0)
Xstd = np.std(X, axis=0)
X = (X-Xmean)/Xstd

y = MMSE_data_df[MMSE_data_df.columns[-1]][:-test_num ]
reg = Lasso(alpha=0.08).fit(X, y)
#reg =ARDRegression().fit(X, y)

print(reg.score(X, y))
y_pred = reg.predict((MMSE_data_df[MMSE_data_df.columns[6:-1]][-test_num:].values-Xmean)/Xstd)
print(y_pred)
y_true = MMSE_data_df[MMSE_data_df.columns[-1]][-test_num: ]
print(list(y_true))
print(r2_score(y_true, y_pred))
print(np.sqrt(mean_squared_error(y_true, y_pred)))

In [ ]:
plt.scatter(y_true, y_pred)
plt.plot([0,30],[0,30])
plt.xlabel('True_Score')
plt.ylabel('Pred_Score')
plt.show()

In [ ]:
seed = 15
kfold = model_selection.KFold(n_splits=10, random_state=seed)
model = Lasso(alpha=0.1)
scoring = 'neg_mean_absolute_error'
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring=scoring)
print(results)
print("MAE:", results.mean(), results.std())
results = model_selection.cross_val_score(model, X, y, cv=kfold, scoring='r2')
print(results)
print("R^2:", results.mean(), results.std())


In [ ]:
MMSE_data_df_filtered

In [ ]:
MMSE_data_df_filtered =  MMSE_data_df[MMSE_data_df['MMSE'] >= 20]
X =  MMSE_data_df_filtered[MMSE_data_df_filtered.columns[6:-1]][:-test_num ].values
# then standardize the features
Xmean = np.mean(X, axis=0)
Xstd = np.std(X, axis=0)
X = (X-Xmean)/Xstd

y = MMSE_data_df_filtered[MMSE_data_df_filtered.columns[-1]][:-test_num ]
reg = Lasso(alpha=0.08).fit(X, y)
#reg =ARDRegression().fit(X, y)

print(reg.score(X, y))
y_pred = reg.predict((MMSE_data_df_filtered[MMSE_data_df_filtered.columns[6:-1]][-test_num:].values-Xmean)/Xstd)
print(y_pred)
y_true = MMSE_data_df_filtered[MMSE_data_df_filtered.columns[-1]][-test_num: ]
print(list(y_true))
print(r2_score(y_true, y_pred))
print(np.sqrt(mean_squared_error(y_true, y_pred)))

In [ ]:
plt.scatter(y_true, y_pred)
plt.plot([0,30],[0,30])
plt.xlabel('True_Score')
plt.ylabel('Pred_Score')
plt.show()

# Predict CDR

In [ ]:
RPDR_CDR_df = pd.read_excel("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\RPDR_CDR_df_Reviewed.xlsx")
EDW_CDR_df = pd.read_excel("${DEMENTIA_DATA_ROOT}\\Brain Age\\final_data\\EDW_CDR_df_Reviewed.xlsx")


### Ordinal linear regression 

# #Haoqi's attempt on MoCA

In [ ]:
Xnames = MoCA_data_df.columns[6:-1]
Xtr =  MoCA_data_df[Xnames][:-test_num ].values
ytr = MoCA_data_df[MoCA_data_df.columns[-1]][:-test_num ].values
Xte =  MoCA_data_df[Xnames][-test_num: ].values
yte = MoCA_data_df[MoCA_data_df.columns[-1]][-test_num: ].values

print(Xtr.shape)
print(ytr)
print(Xte.shape)
print(yte)

In [ ]:
from scipy.stats import pearsonr
from sklearn.linear_model import BayesianRidge, ARDRegression
from sklearn.svm import SVR

# first do simple univariate feature selection
corrs = []
for i in range(Xtr.shape[1]):
    corrs.append(np.abs(pearsonr(Xtr[:,i], ytr)[0]))
corrs = np.array(corrs)
    
K = 10
good_ids = np.argsort(-corrs)[:K]
X2 = Xtr[:, good_ids]

# then standardize the features
Xmean = np.mean(X2, axis=0)
Xstd = np.std(X2, axis=0)
X3 = (X2-Xmean)/Xstd

# fit the model
reg = BayesianRidge()
reg.fit(X3, ytr)

print(pd.DataFrame(data=np.c_[Xnames[good_ids], reg.coef_], columns=['Feature', 'Coef']))

# testing
Xte2 = (Xte[:, good_ids]-Xmean)/Xstd
ypte = reg.predict(Xte2)

print(pearsonr(yte, ypte))
print(r2_score(yte, ypte))
plt.scatter(yte, ypte)
plt.plot([0,30],[0,30])
plt.xlabel('True_Score')
plt.ylabel('Pred_Score')
plt.show()

In [ ]:
X2 = Xtr[:, good_ids]
X2